# LeetCode 1925 — Count Square Sum Triples bằng Python

Link đề: https://leetcode.com/problems/count-square-sum-triples/

Độ khó: **Easy** · Chủ đề: Math, Enumeration

```python
class Solution:
    def countTriples(self, n: int) -> int:

```

**Mục tiêu học:**

1. Đọc đúng đề **đếm bộ có thứ tự**: `(3,4,5)` và `(4,3,5)` là hai bộ khác nhau.
2. Brute force 3 vòng lặp `O(n³)`, rồi cải tiến: cố định `(a, b)`, **suy ra** `c` → `O(n²)`.
3. Kiểm tra "số chính phương" đúng cách bằng `math.isqrt` (tránh sai số của `sqrt` số thực), hoặc tra cứu bằng `set`.
4. Kỹ thuật **hai con trỏ** cho mỗi `c` và công thức **Euclid** sinh bộ ba Pythagore — lời giải sáng tạo nhất.
5. So sánh 5 cách, kiểm thử đối chiếu, nhận dạng pattern "bớt một vòng lặp bằng cách tính ra biến cuối".

## 1. Bài toán yêu cầu gì?

Một **square triple** `(a, b, c)` là bộ số nguyên thỏa `a² + b² = c²`.

Cho `n`, đếm số bộ `(a, b, c)` với `1 <= a, b, c <= n`.

| n | Kết quả | Các bộ |
|---:|---:|---|
| 5 | 2 | `(3,4,5)`, `(4,3,5)` |
| 10 | 4 | `(3,4,5)`, `(4,3,5)`, `(6,8,10)`, `(8,6,10)` |
| 4 | 0 | chưa có bộ nào |

**Ràng buộc:** `1 <= n <= 250`.

Chú ý: đề đếm **có thứ tự** giữa `a` và `b`. Và `c` luôn lớn hơn cả `a` lẫn `b` (vì `c² = a² + b² > a²`).

In [1]:
examples = [(5, 2), (10, 4), (4, 0), (1, 0)]
for n, expected in examples:
    print(f"n = {n:3} -> kỳ vọng {expected}")

n =   5 -> kỳ vọng 2
n =  10 -> kỳ vọng 4
n =   4 -> kỳ vọng 0
n =   1 -> kỳ vọng 0


## 2. Nền tảng Python cần biết

### 2.1. Lũy thừa `**` và bình phương

`a ** 2` là `a` bình phương; `a * a` cho cùng kết quả (thường nhanh hơn một chút).

In [2]:
a, b_ = 3, 4
print(a ** 2 + b_ ** 2, a * a + b_ * b_, 5 ** 2)
print(3 ** 2 + 4 ** 2 == 5 ** 2)

25 25 25
True


### 2.2. Căn bậc hai: `math.sqrt` (số thực) và `math.isqrt` (số nguyên)

- `math.sqrt(x)` trả về **số thực** (`float`) — có thể có sai số làm tròn với số lớn.
- `math.isqrt(x)` trả về **phần nguyên** của căn, tức số nguyên `r` lớn nhất với `r*r <= x`. Luôn chính xác.

`x` là **số chính phương** ⇔ `isqrt(x) ** 2 == x`.

In [3]:
import math

print(math.sqrt(25), math.sqrt(26))
print(math.isqrt(25), math.isqrt(26), math.isqrt(24))
for x in [25, 26, 0, 1, 144, 145]:
    r = math.isqrt(x)
    print(f"{x}: isqrt = {r}, chính phương? {r * r == x}")

5.0 5.0990195135927845
5 5 4
25: isqrt = 5, chính phương? True
26: isqrt = 5, chính phương? False
0: isqrt = 0, chính phương? True
1: isqrt = 1, chính phương? True
144: isqrt = 12, chính phương? True
145: isqrt = 12, chính phương? False


### 2.3. Vì sao không nên dùng `sqrt(x) == int(sqrt(x))`?

Với số nhỏ như trong bài (≤ 2·250² = 125.000) thì `float` vẫn đúng, nhưng với số lớn, `float` chỉ có ~15–16 chữ số chính xác → có thể **nhận nhầm** một số không chính phương là chính phương.

In [4]:
big = (10**8 + 1) ** 2 - 1          # KHÔNG phải số chính phương
print("float nói chính phương?", math.sqrt(big) == int(math.sqrt(big)))
print("isqrt nói chính phương?", math.isqrt(big) ** 2 == big)

float nói chính phương? True
isqrt nói chính phương? False


### 2.4. `set` các số chính phương — tra cứu O(1) trung bình

Tạo trước `squares = {c*c for c in range(1, n+1)}` (set comprehension), rồi hỏi `x in squares`.

In [5]:
n = 10
squares = {c * c for c in range(1, n + 1)}
print(sorted(squares))
print(25 in squares, 26 in squares)

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
True False


### 2.5. `math.gcd` — ước chung lớn nhất

Dùng ở cách Euclid: hai số **nguyên tố cùng nhau** khi `gcd = 1`.

In [6]:
print(math.gcd(12, 18), math.gcd(7, 5), math.gcd(4, 2))

6 1 2


## 3. Tính bằng tay

Với `n = 10`, các `c` khả dĩ có `c² = a² + b²`:

| c | c² | Tách thành a² + b² (1 ≤ a, b ≤ 10) |
|---:|---:|---|
| 5 | 25 | 9 + 16 → (3,4), (4,3) |
| 10 | 100 | 36 + 64 → (6,8), (8,6) |
| các `c` khác | | không tách được |

Tổng: **4** bộ.

Quan sát:

- Nghiệm luôn đi thành **cặp** `(a,b)` và `(b,a)` vì `a ≠ b` (nếu `a = b` thì `c = a·√2` không nguyên).
- `(6,8,10)` là `(3,4,5)` nhân 2 — nghiệm "gốc" sinh ra nghiệm "bội".

## 4. Cách 1 — Brute force ba vòng lặp

Thử mọi `(a, b, c)` trong `[1, n]³`.

In [7]:
def count_triples_brute(n):
    count = 0
    for a in range(1, n + 1):
        for b in range(1, n + 1):
            for c in range(1, n + 1):
                if a * a + b * b == c * c:
                    count += 1
    return count

for n, expected in examples:
    print(n, "->", count_triples_brute(n), "| kỳ vọng", expected)

5 -> 2 | kỳ vọng 2
10 -> 4 | kỳ vọng 4
4 -> 0 | kỳ vọng 0
1 -> 0 | kỳ vọng 0


**Độ phức tạp:** `O(n³)`. Với `n = 250` là `250³ ≈ 15,6 triệu` lần lặp — Python mất vài giây, có thể bị Time Limit.

**Nút thắt:** khi đã biết `a` và `b`, giá trị `c` **đã bị xác định** (`c = √(a² + b²)`). Vòng lặp `c` thứ ba chỉ đi "tìm" một số mà ta có thể **tính** ra.

## 5. Cách 2 — Cố định `(a, b)`, tính `c` bằng `isqrt`

1. Với mỗi `a, b`: `s = a*a + b*b`.
2. `c = isqrt(s)`.
3. Nếu `c*c == s` **và** `c <= n` → đếm.

In [8]:
def count_triples_isqrt(n):
    count = 0
    for a in range(1, n + 1):
        for b in range(1, n + 1):
            s = a * a + b * b
            c = math.isqrt(s)
            if c * c == s and c <= n:
                count += 1
    return count

for n, expected in examples:
    print(n, "->", count_triples_isqrt(n), "| kỳ vọng", expected)

5 -> 2 | kỳ vọng 2
10 -> 4 | kỳ vọng 4
4 -> 0 | kỳ vọng 0
1 -> 0 | kỳ vọng 0


**Độ phức tạp:** `O(n²)` lần lặp (mỗi `isqrt` coi như `O(1)` với số nhỏ) → 62.500 lần với `n = 250`. Bộ nhớ `O(1)`.

Đừng quên điều kiện `c <= n`: ví dụ `n = 4`, `(3,4)` cho `c = 5 > 4` → **không** đếm.

## 6. Cách 3 — Tra cứu trong `set` các số chính phương

In [9]:
def count_triples_set(n):
    squares = {c * c for c in range(1, n + 1)}     # chỉ chứa c <= n -> tự lọc điều kiện c <= n
    count = 0
    for a in range(1, n + 1):
        for b in range(1, n + 1):
            if a * a + b * b in squares:
                count += 1
    return count

for n, expected in examples:
    print(n, "->", count_triples_set(n), "| kỳ vọng", expected)

5 -> 2 | kỳ vọng 2
10 -> 4 | kỳ vọng 4
4 -> 0 | kỳ vọng 0
1 -> 0 | kỳ vọng 0


Điểm hay: vì `squares` chỉ chứa bình phương của `1..n`, phép `in squares` kiểm tra **cùng lúc** "là số chính phương" và "`c <= n`".

**Tối ưu nhỏ:** chỉ đếm `a < b` rồi nhân 2 (vì nghiệm đi theo cặp, `a ≠ b`). Giảm một nửa số lần lặp; thêm nữa có thể dừng sớm khi `a*a + b*b > n*n`.

In [10]:
def count_triples_half(n):
    nn = n * n
    squares = {c * c for c in range(1, n + 1)}
    count = 0
    for a in range(1, n + 1):
        for b in range(a + 1, n + 1):
            s = a * a + b * b
            if s > nn:
                break                 # b tăng thì s tăng -> không còn c <= n
            if s in squares:
                count += 2            # (a, b, c) và (b, a, c)
    return count

for n, expected in examples:
    print(n, "->", count_triples_half(n), "| kỳ vọng", expected)

5 -> 2 | kỳ vọng 2
10 -> 4 | kỳ vọng 4
4 -> 0 | kỳ vọng 0
1 -> 0 | kỳ vọng 0


## 7. Lời giải chính — Giảng từng đoạn rồi ghép (Cách 3, bản đầy đủ)

**Đoạn 1 — chuẩn bị tập bình phương:** `squares = {c*c for c in range(1, n+1)}`. Tốn `O(n)`.

**Đoạn 2 — duyệt mọi cặp có thứ tự `(a, b)`:** hai vòng `for` từ 1 tới `n`.

**Đoạn 3 — kiểm tra:** `a*a + b*b in squares`.

Ta chọn bản đếm đủ cặp có thứ tự (không nhân 2) làm lời giải chính vì nó **bám sát đề nhất** — ít chỗ sai nhất cho người mới. Bản `a < b` ×2 là tối ưu thêm, đã được kiểm chứng ở trên.

## 8. TRACE từng bước

Trace với `n = 10`, chỉ in những cặp `(a, b)` mà `a² + b²` "gần" là chính phương để dễ quan sát: in mọi cặp có `a*a + b*b <= n*n`, kèm quyết định.

In [11]:
def trace_count_triples(n, show_limit=200):
    squares = {c * c for c in range(1, n + 1)}
    print("squares =", sorted(squares))
    count = 0
    shown = 0
    for a in range(1, n + 1):
        for b in range(1, n + 1):
            s = a * a + b * b
            hit = s in squares
            if hit:
                count += 1
            if s <= n * n and (hit or a == 3) and shown < show_limit:
                tag = f"= {math.isqrt(s)}² -> ĐẾM (count = {count})" if hit else "không chính phương"
                print(f"  a={a}, b={b}: {a}²+{b}² = {s} {tag}")
                shown += 1
    print("Kết quả:", count)
    return count

trace_count_triples(10)

squares = [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
  a=3, b=1: 3²+1² = 10 không chính phương
  a=3, b=2: 3²+2² = 13 không chính phương
  a=3, b=3: 3²+3² = 18 không chính phương
  a=3, b=4: 3²+4² = 25 = 5² -> ĐẾM (count = 1)
  a=3, b=5: 3²+5² = 34 không chính phương
  a=3, b=6: 3²+6² = 45 không chính phương
  a=3, b=7: 3²+7² = 58 không chính phương
  a=3, b=8: 3²+8² = 73 không chính phương
  a=3, b=9: 3²+9² = 90 không chính phương
  a=4, b=3: 4²+3² = 25 = 5² -> ĐẾM (count = 2)
  a=6, b=8: 6²+8² = 100 = 10² -> ĐẾM (count = 3)
  a=8, b=6: 8²+6² = 100 = 10² -> ĐẾM (count = 4)
Kết quả: 4


4

Hàng `a = 3` được in đầy đủ để thấy các cặp **không** đếm; với `a` khác chỉ in các cặp trúng. Ta thấy đúng 4 lần đếm, khớp bảng tính tay.

### Trace dạng bảng theo `c` (nhìn từ phía `c`)

In [12]:
def triples_by_c(n):
    for c in range(1, n + 1):
        pairs = [(a, b) for a in range(1, c) for b in range(1, c) if a * a + b * b == c * c]
        if pairs:
            print(f"c = {c:3}: {pairs}")

triples_by_c(30)

c =   5: [(3, 4), (4, 3)]
c =  10: [(6, 8), (8, 6)]
c =  13: [(5, 12), (12, 5)]
c =  15: [(9, 12), (12, 9)]
c =  17: [(8, 15), (15, 8)]
c =  20: [(12, 16), (16, 12)]
c =  25: [(7, 24), (15, 20), (20, 15), (24, 7)]
c =  26: [(10, 24), (24, 10)]
c =  29: [(20, 21), (21, 20)]
c =  30: [(18, 24), (24, 18)]


## 9. Cách 4 — Hai con trỏ cho mỗi `c`

Cố định `c`. Tìm số cặp `a < b` với `a² + b² = c²`, `a, b` trong `[1, c-1]`:

- `lo = 1`, `hi = c - 1`.
- Nếu `lo² + hi² == c²` → tìm thấy, `lo += 1`, `hi -= 1`.
- Nếu `< c²` → cần lớn hơn → `lo += 1`.
- Nếu `> c²` → cần nhỏ hơn → `hi -= 1`.

Mỗi `c` tốn `O(c)` → tổng `O(n²)`, bộ nhớ `O(1)`, không cần `isqrt` hay `set`.

In [13]:
def count_triples_two_pointers(n):
    count = 0
    for c in range(1, n + 1):
        target = c * c
        lo, hi = 1, c - 1
        while lo < hi:
            s = lo * lo + hi * hi
            if s == target:
                count += 2            # (lo, hi) và (hi, lo)
                lo += 1
                hi -= 1
            elif s < target:
                lo += 1
            else:
                hi -= 1
    return count

for n, expected in examples:
    print(n, "->", count_triples_two_pointers(n), "| kỳ vọng", expected)

5 -> 2 | kỳ vọng 2
10 -> 4 | kỳ vọng 4
4 -> 0 | kỳ vọng 0
1 -> 0 | kỳ vọng 0


In [14]:
def trace_two_pointers(c):
    target = c * c
    lo, hi = 1, c - 1
    print(f"c = {c}, c² = {target}")
    while lo < hi:
        s = lo * lo + hi * hi
        act = "TÌM THẤY" if s == target else ("nhỏ -> lo += 1" if s < target else "lớn -> hi -= 1")
        print(f"  lo={lo:2}, hi={hi:2}: {lo}²+{hi}² = {s:4} -> {act}")
        if s == target:
            lo += 1; hi -= 1
        elif s < target:
            lo += 1
        else:
            hi -= 1

trace_two_pointers(5)
print()
trace_two_pointers(13)

c = 5, c² = 25
  lo= 1, hi= 4: 1²+4² =   17 -> nhỏ -> lo += 1
  lo= 2, hi= 4: 2²+4² =   20 -> nhỏ -> lo += 1
  lo= 3, hi= 4: 3²+4² =   25 -> TÌM THẤY

c = 13, c² = 169
  lo= 1, hi=12: 1²+12² =  145 -> nhỏ -> lo += 1
  lo= 2, hi=12: 2²+12² =  148 -> nhỏ -> lo += 1
  lo= 3, hi=12: 3²+12² =  153 -> nhỏ -> lo += 1
  lo= 4, hi=12: 4²+12² =  160 -> nhỏ -> lo += 1
  lo= 5, hi=12: 5²+12² =  169 -> TÌM THẤY
  lo= 6, hi=11: 6²+11² =  157 -> nhỏ -> lo += 1
  lo= 7, hi=11: 7²+11² =  170 -> lớn -> hi -= 1
  lo= 7, hi=10: 7²+10² =  149 -> nhỏ -> lo += 1
  lo= 8, hi=10: 8²+10² =  164 -> nhỏ -> lo += 1
  lo= 9, hi=10: 9²+10² =  181 -> lớn -> hi -= 1


## 10. Cách 5 (sáng tạo) — Công thức Euclid sinh bộ ba Pythagore

Mọi bộ ba **nguyên thủy** (primitive, `gcd(a,b,c) = 1`) có dạng, với `p > q >= 1`, `gcd(p, q) = 1`, `p - q` lẻ:

```text
a = p² - q²,   b = 2pq,   c = p² + q²
```

Mọi bộ ba khác là **bội** `k·(a, b, c)` của một bộ nguyên thủy. Vậy: duyệt `(p, q)` sinh bộ nguyên thủy với `c <= n`, rồi đếm số bội `k = n // c`. Mỗi bộ cho 2 thứ tự `(a,b)`, `(b,a)`.

Ví dụ `p = 2, q = 1`: `(3, 4, 5)`. `p = 3, q = 2`: `(5, 12, 13)`. `p = 4, q = 1`: `(15, 8, 17)`.

In [15]:
def primitive_triples(limit):
    out = []
    p = 2
    while p * p + 1 <= limit:
        for q in range(1, p):
            if (p - q) % 2 == 1 and math.gcd(p, q) == 1:
                c = p * p + q * q
                if c <= limit:
                    out.append((p * p - q * q, 2 * p * q, c))
        p += 1
    return out

print(primitive_triples(30))

def count_triples_euclid(n):
    total = 0
    for a, b_, c in primitive_triples(n):
        total += 2 * (n // c)        # k = 1..n//c, mỗi k có 2 thứ tự
    return total

for n, expected in examples:
    print(n, "->", count_triples_euclid(n), "| kỳ vọng", expected)

[(3, 4, 5), (5, 12, 13), (15, 8, 17), (7, 24, 25), (21, 20, 29)]
5 -> 2 | kỳ vọng 2
10 -> 4 | kỳ vọng 4
4 -> 0 | kỳ vọng 0
1 -> 0 | kỳ vọng 0


Số cặp `(p, q)` với `p² + q² <= n` là khoảng `O(n)` (vì `p <= √n`, `q < p`), nên cách này rất nhanh — nhưng cần hiểu một định lý số học, nên chỉ là "phần mở rộng".

## 11. Code nộp LeetCode — lời giải chính

In [16]:
class Solution:
    def countTriples(self, n: int) -> int:
        squares = {c * c for c in range(1, n + 1)}
        count = 0
        for a in range(1, n + 1):
            for b in range(1, n + 1):
                if a * a + b * b in squares:
                    count += 1
        return count


sol = Solution()
for n, expected in examples:
    assert sol.countTriples(n) == expected
print("Solution (set) PASS")

Solution (set) PASS


In [17]:
class SolutionIsqrt:
    def countTriples(self, n: int) -> int:
        count = 0
        for a in range(1, n + 1):
            for b in range(a + 1, n + 1):
                s = a * a + b * b
                c = math.isqrt(s)
                if c > n:
                    break
                if c * c == s:
                    count += 2
        return count


for n, expected in examples:
    assert SolutionIsqrt().countTriples(n) == expected
print("SolutionIsqrt PASS")

SolutionIsqrt PASS


Trên LeetCode, `math` đã được import sẵn; chạy ngoài LeetCode thì nhớ `import math`.

## 12. Vì sao đúng và độ phức tạp

**Cách 2/3 đúng:** với mỗi cặp có thứ tự `(a, b)` trong `[1, n]²` tồn tại **nhiều nhất một** `c >= 1` thỏa `c² = a² + b²` (căn dương là duy nhất). Vậy số bộ `(a, b, c)` = số cặp `(a, b)` mà `a² + b²` là bình phương của một số trong `[1, n]` — chính là điều ta đếm.

**Tối ưu `a < b`, ×2 đúng:** `a = b` không bao giờ là nghiệm (`2a² = c²` ⇒ `√2 = c/a` hữu tỉ — vô lý). Phép đổi chỗ `(a,b) ↔ (b,a)` ghép mọi nghiệm thành cặp. `break` đúng vì với `a` cố định, `a² + b²` tăng theo `b`.

**Hai con trỏ đúng:** với `c` cố định, `f(lo, hi) = lo² + hi²` tăng theo `lo`, tăng theo `hi`. Khi `f < c²`, mọi cặp `(lo, h)` với `h <= hi` đều `< c²` → bỏ `lo` an toàn; tương tự khi `f > c²` bỏ `hi`. (Giống Two Sum II trên mảng đã sắp.)

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|:---:|:---:|
| 1. Ba vòng lặp | thử mọi `(a,b,c)` | `O(n³)` | `O(1)` |
| 2. Hai vòng + `isqrt` | tính `c` | `O(n²)` | `O(1)` |
| 3. Hai vòng + `set` | tra cứu bình phương | `O(n²)` | `O(n)` |
| 4. Hai con trỏ theo `c` | thu hẹp `lo, hi` | `O(n²)` | `O(1)` |
| 5. Euclid | sinh bộ nguyên thủy + bội | ~`O(n)` cặp `(p,q)` | `O(1)`–`O(n)` |

## 13. Bộ test kiểm tra

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:26} PASS")

solvers = [count_triples_brute, count_triples_isqrt, count_triples_set, count_triples_half,
           count_triples_two_pointers, count_triples_euclid]

tests = [
    ("n = 1", 1, 0),
    ("n = 4 (chưa tới 5)", 4, 0),
    ("ví dụ 1: n = 5", 5, 2),
    ("ví dụ 2: n = 10", 10, 4),
    ("n = 13 (thêm 5,12,13)", 13, 6),
    ("n = 15 (thêm 9,12,15)", 15, 8),
    ("n = 25", 25, 16),
]
for name, n, e in tests:
    for f in solvers:
        assert f(n) == e, (name, f.__name__)
    check(name + " (6 cách)", count_triples_set(n), e)

n = 1 (6 cách)             PASS
n = 4 (chưa tới 5) (6 cách) PASS
ví dụ 1: n = 5 (6 cách)    PASS
ví dụ 2: n = 10 (6 cách)   PASS
n = 13 (thêm 5,12,13) (6 cách) PASS
n = 15 (thêm 9,12,15) (6 cách) PASS
n = 25 (6 cách)            PASS


In [19]:
# Đối chiếu các cách nhanh với brute force trên mọi n từ 1 tới 60
for n in range(1, 61):
    expected = count_triples_brute(n)
    for f in solvers[1:]:
        assert f(n) == expected, (n, f.__name__)
print("Khớp brute force với mọi n = 1..60")

# Các cách nhanh khớp nhau với n lớn
big_results = {f.__name__: f(250) for f in solvers[1:]}
print(big_results)
assert len(set(big_results.values())) == 1

Khớp brute force với mọi n = 1..60
{'count_triples_isqrt': 330, 'count_triples_set': 330, 'count_triples_half': 330, 'count_triples_two_pointers': 330, 'count_triples_euclid': 330}


Giải thích `n = 25` → 16: các bộ `a < b` với `c <= 25` là `(3,4,5), (6,8,10), (5,12,13), (9,12,15), (8,15,17), (12,16,20), (15,20,25), (7,24,25)` — 8 bộ, nhân 2 thứ tự = 16.

## 14. Lỗi người mới hay gặp

### Lỗi 1: Chỉ đếm `a <= b` mà quên nhân 2

In [20]:
def wrong_unordered(n):
    squares = {c * c for c in range(1, n + 1)}
    return sum(1 for a in range(1, n + 1) for b in range(a, n + 1) if a * a + b * b in squares)

print("Sai :", wrong_unordered(10), "| Đúng:", count_triples_set(10))

Sai : 2 | Đúng: 4


### Lỗi 2: Quên điều kiện `c <= n` khi dùng `isqrt`

In [21]:
def wrong_no_c_limit(n):
    count = 0
    for a in range(1, n + 1):
        for b in range(1, n + 1):
            s = a * a + b * b
            if math.isqrt(s) ** 2 == s:        # SAI: thiếu c <= n
                count += 1
    return count

print("Sai :", wrong_no_c_limit(4), "| Đúng:", count_triples_isqrt(4), "  (3,4,5) có c = 5 > 4)")

Sai : 2 | Đúng: 0   (3,4,5) có c = 5 > 4)


### Lỗi 3: Dùng `int(math.sqrt(s))` và so sánh float

```python
c = math.sqrt(s)
if c == int(c): ...      # rủi ro sai số với số lớn
```

Trong bài (`s <= 125.000`) vẫn đúng, nhưng là thói quen nguy hiểm (xem mục 2.3). Dùng `math.isqrt`.

### Lỗi 4: Vòng lặp `range(1, n)` bỏ mất `n`

In [22]:
def wrong_range(n):
    squares = {c * c for c in range(1, n)}        # SAI: thiếu c = n
    return sum(1 for a in range(1, n) for b in range(1, n) if a * a + b * b in squares)

print("Sai :", wrong_range(5), "| Đúng:", count_triples_set(5))

Sai : 0 | Đúng: 2


### Lỗi 5: `break` sai chỗ trong bản tối ưu

Nếu `break` khi `s` **không** chính phương (thay vì khi `s > n*n`) thì bỏ sót rất nhiều nghiệm. Chỉ được dừng khi chắc chắn mọi `b` lớn hơn đều vượt giới hạn.

## 15. Mẹo nhận dạng pattern và biến thể

- "Đếm bộ k phần tử thỏa một **phương trình**" → cố định `k-1` biến, **tính** biến cuối (hoặc tra `set`) → giảm một bậc độ phức tạp.
- Phương trình đơn điệu theo từng biến → **hai con trỏ**.
- Bài số học về bộ ba Pythagore → công thức Euclid, `gcd`.

Bài liên quan: 633 (Sum of Square Numbers — hai con trỏ / `isqrt`), 1 (Two Sum), 15 (3Sum), 167 (Two Sum II), 367 (Valid Perfect Square).

### Biến thể: LeetCode 633 — `c` có viết được thành `a² + b²` (a, b >= 0)?

In [23]:
def judge_square_sum(c):
    lo, hi = 0, math.isqrt(c)
    while lo <= hi:
        s = lo * lo + hi * hi
        if s == c:
            return True
        if s < c:
            lo += 1
        else:
            hi -= 1
    return False

print([x for x in range(30) if judge_square_sum(x)])

[0, 1, 2, 4, 5, 8, 9, 10, 13, 16, 17, 18, 20, 25, 26, 29]


## 16. Bài tập tự luyện (TODO)

### Bài 1 — Tính tay

Không chạy code: `countTriples(12)`, `countTriples(17)`, `countTriples(20)`? Ghi vào `ex1_answers`.

In [24]:
ex1_answers = None   # TODO: list 3 số

### Bài 2 — Điền chỗ trống: kiểm tra số chính phương bằng `isqrt`

In [25]:
def is_perfect_square(x: int) -> bool:
    # TODO
    pass

# assert is_perfect_square(0) and is_perfect_square(1) and is_perfect_square(144)
# assert not is_perfect_square(2) and not is_perfect_square(145)

### Bài 3 — Liệt kê các bộ (không chỉ đếm)

Trả về list các bộ `(a, b, c)` có thứ tự, sắp theo `(c, a)`.

In [26]:
def list_triples(n):
    # TODO
    pass

# assert list_triples(10) == [(3, 4, 5), (4, 3, 5), (6, 8, 10), (8, 6, 10)]

### Bài 4 — Tự viết lời giải hai con trỏ theo `c`

In [27]:
def ex4_count_triples(n):
    # TODO
    pass

# for n, e in examples: assert ex4_count_triples(n) == e

### Bài 5 — Biến thể: đếm bộ **không có thứ tự** với `a <= b`, và cho phép `a, b` từ 0

Đếm số cặp `(a, b)`, `0 <= a <= b`, `a² + b² = c²` với `1 <= c <= n` (khi `a = 0` thì `b = c`).

In [28]:
def count_with_zero(n):
    # TODO
    pass

# assert count_with_zero(5) == 6      # (0,c) cho c=1..5, và (3,4)

### Bài 6 — Mini challenge: chỉ đếm bộ **nguyên thủy**

Đếm số bộ có thứ tự `(a, b, c)`, `c <= n`, với `gcd(a, b) = 1`. Đối chiếu với công thức Euclid.

In [29]:
def count_primitive(n):
    # TODO
    pass

# assert count_primitive(10) == 2
# assert count_primitive(30) == 2 * len(primitive_triples(30))

## 17. Gợi ý 3 mức

**Bài 1.** (1) Liệt kê bộ `a < b`. (2) Với `n = 17` có thêm `(5,12,13)`, `(9,12,15)`, `(8,15,17)`. (3) Nhân 2.

**Bài 2.** (1) Căn nguyên. (2) `r = math.isqrt(x)`. (3) `return r * r == x`.

**Bài 3.** (1) Dùng hai vòng và `set`. (2) Lưu `(a, b, isqrt(s))`. (3) `sorted(..., key=lambda t: (t[2], t[0]))`.

**Bài 4.** (1) Với mỗi `c`, `lo = 1, hi = c - 1`. (2) `while lo < hi`. (3) Trúng thì `count += 2`, dịch cả hai.

**Bài 5.** (1) `a = 0` luôn cho một cặp `(0, c)`. (2) Thêm các cặp `1 <= a < b`. (3) `n + (số bộ a<b)`.

**Bài 6.** (1) Lời giải `set` + điều kiện `math.gcd(a, b) == 1`. (2) Bộ nguyên thủy không có bội số chung. (3) So với `2 * len(primitive_triples(n))`.

## 18. Đáp án tham khảo + template nhớ nhanh

In [30]:
ans_ex1 = [4, 10, 12]


def ans_is_perfect_square(x):
    r = math.isqrt(x)
    return r * r == x


def ans_list_triples(n):
    squares = {c * c for c in range(1, n + 1)}
    out = []
    for a in range(1, n + 1):
        for b in range(1, n + 1):
            s = a * a + b * b
            if s in squares:
                out.append((a, b, math.isqrt(s)))
    return sorted(out, key=lambda t: (t[2], t[0]))


def ans_ex4_count_triples(n):
    count = 0
    for c in range(1, n + 1):
        lo, hi = 1, c - 1
        while lo < hi:
            s = lo * lo + hi * hi
            if s == c * c:
                count += 2; lo += 1; hi -= 1
            elif s < c * c:
                lo += 1
            else:
                hi -= 1
    return count


def ans_count_with_zero(n):
    squares = {c * c for c in range(1, n + 1)}
    extra = sum(1 for a in range(1, n + 1) for b in range(a + 1, n + 1) if a * a + b * b in squares)
    return n + extra


def ans_count_primitive(n):
    squares = {c * c for c in range(1, n + 1)}
    return sum(1 for a in range(1, n + 1) for b in range(1, n + 1)
               if a * a + b * b in squares and math.gcd(a, b) == 1)


assert ans_ex1 == [count_triples_set(x) for x in [12, 17, 20]]
assert ans_is_perfect_square(0) and ans_is_perfect_square(1) and ans_is_perfect_square(144)
assert not ans_is_perfect_square(2) and not ans_is_perfect_square(145)
assert ans_list_triples(10) == [(3, 4, 5), (4, 3, 5), (6, 8, 10), (8, 6, 10)]
for n, e in examples:
    assert ans_ex4_count_triples(n) == e
assert ans_count_with_zero(5) == 6
assert ans_count_primitive(10) == 2
for n in range(1, 120):
    assert ans_count_primitive(n) == 2 * len(primitive_triples(n)), n
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


### Template nhớ nhanh

```python
class Solution:
    def countTriples(self, n: int) -> int:
        squares = {c * c for c in range(1, n + 1)}   # tự lọc c <= n
        return sum(1 for a in range(1, n + 1)
                     for b in range(1, n + 1)
                     if a * a + b * b in squares)
```

**Khung tư duy:** "biến cuối bị xác định bởi các biến trước" → đừng lặp nó, hãy **tính** hoặc **tra cứu** nó.

### Checklist trước khi submit

- [ ] Đếm **có thứ tự** (`(3,4,5)` và `(4,3,5)`)? Nếu chỉ duyệt `a < b` thì đã nhân 2?
- [ ] Có điều kiện `c <= n`?
- [ ] Kiểm tra chính phương bằng `isqrt` hoặc `set`, không so sánh `float`?
- [ ] `range(1, n + 1)` gồm cả `n`?
- [ ] Tên method `countTriples`, trả `int`?